# 13. Game Theory and Decision Theory

Game Theory studies strategic interactions where one player's outcome depends on others' decisions.

**Why Game Theory Matters for Calculus:**
- Nash equilibrium found by solving $\frac{\partial U_i}{\partial x_i} = 0$ for all players
- Best response functions use derivatives to maximize utility
- Mixed strategy equilibria involve optimization with constraints
- Expected utility maximization uses integration over probability distributions
- Evolutionary game theory uses differential equations

**Topics Covered:**
1. Normal form games and dominant strategies
2. Nash equilibrium (pure and mixed strategies)
3. Decision theory under uncertainty
4. Expected utility theory
5. Continuous strategy games (using calculus!)
6. Evolutionary game theory
7. Applications: auctions, bargaining, mechanism design

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import optimize
from scipy.optimize import minimize, linprog
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import Rectangle

sns.set_style('whitegrid')
np.random.seed(42)

## 1. Introduction to Game Theory - Normal Form Games

A **normal form game** consists of:
- Players: $N = \{1, 2, \ldots, n\}$
- Strategies: $S_i$ for each player $i$
- Payoffs: $u_i(s_1, s_2, \ldots, s_n)$ for each strategy profile

**Classic Example: Prisoner's Dilemma**

Two criminals are arrested. Each can either Cooperate (stay silent) or Defect (betray).

In [ ]:
# Prisoner's Dilemma payoff matrix
# Rows: Player 1's strategies, Columns: Player 2's strategies
# Each cell: (Player 1 payoff, Player 2 payoff)

pd_payoffs = {
    ('Cooperate', 'Cooperate'): (-1, -1),
    ('Cooperate', 'Defect'): (-3, 0),
    ('Defect', 'Cooperate'): (0, -3),
    ('Defect', 'Defect'): (-2, -2)
}

# Create payoff matrices
strategies = ['Cooperate', 'Defect']
P1_payoffs = np.array([[-1, -3], [0, -2]])  # Player 1's payoffs
P2_payoffs = np.array([[-1, 0], [-3, -2]])  # Player 2's payoffs

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Player 1's payoff matrix
im1 = axes[0].imshow(P1_payoffs, cmap='RdYlGn', aspect='auto', vmin=-3, vmax=0)
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['P2: Cooperate', 'P2: Defect'])
axes[0].set_yticklabels(['P1: Cooperate', 'P1: Defect'])
axes[0].set_title('Player 1 Payoffs', fontsize=14, fontweight='bold')

# Add text annotations
for i in range(2):
    for j in range(2):
        text = axes[0].text(j, i, P1_payoffs[i, j],
                           ha="center", va="center", color="black", fontsize=20, fontweight='bold')

plt.colorbar(im1, ax=axes[0])

# Combined payoff table
axes[1].axis('off')
table_data = []
for i, s1 in enumerate(strategies):
    row = [s1]
    for j, s2 in enumerate(strategies):
        row.append(f"({P1_payoffs[i,j]}, {P2_payoffs[i,j]})")
    table_data.append(row)

table = axes[1].table(cellText=table_data,
                     colLabels=['Player 1 \\ Player 2', 'Cooperate', 'Defect'],
                     cellLoc='center',
                     loc='center',
                     bbox=[0.1, 0.3, 0.8, 0.4])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Highlight Nash equilibrium
table[(2, 2)].set_facecolor('#ff9999')

analysis_text = """Prisoner's Dilemma Analysis:

Payoffs: (Player 1, Player 2)
Lower numbers = worse outcome (years in prison)

Dominant Strategy Analysis:
• If P2 cooperates: P1 gets -1 (coop) vs 0 (defect) → Defect better
• If P2 defects: P1 gets -3 (coop) vs -2 (defect) → Defect better
• Defect dominates Cooperate for P1 (by symmetry, also for P2)

Nash Equilibrium: (Defect, Defect) [shown in red]
• Neither player can improve by unilaterally changing strategy
• Payoff: (-2, -2)

Paradox:
• Both would be better off at (Cooperate, Cooperate): (-1, -1)
• But cooperation is not stable!
• Classic example of individual rationality vs collective good
"""

axes[1].text(0.5, 0.05, analysis_text, fontsize=9, family='monospace',
             verticalalignment='bottom', horizontalalignment='center',
             transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

print(analysis_text)

## 2. Nash Equilibrium - Pure Strategies

**Definition:** A strategy profile $(s_1^*, s_2^*, \ldots, s_n^*)$ is a **Nash equilibrium** if no player can improve their payoff by unilaterally changing their strategy:

$$u_i(s_1^*, \ldots, s_i^*, \ldots, s_n^*) \geq u_i(s_1^*, \ldots, s_i, \ldots, s_n^*) \quad \forall s_i \in S_i, \forall i$$

**Example: Battle of the Sexes**

A couple wants to spend time together but have different preferences.

In [ ]:
# Battle of the Sexes
# Two Nash equilibria in pure strategies!

strategies_bos = ['Opera', 'Football']
P1_bos = np.array([[2, 0], [0, 1]])  # Player 1 prefers Opera
P2_bos = np.array([[1, 0], [0, 2]])  # Player 2 prefers Football

def find_nash_pure(P1, P2):
    """Find pure strategy Nash equilibria."""
    n_strategies = P1.shape[0]
    nash_equilibria = []
    
    for i in range(n_strategies):
        for j in range(n_strategies):
            # Check if (i, j) is a Nash equilibrium
            # Player 1 cannot improve by changing from i
            p1_cannot_improve = all(P1[i, j] >= P1[k, j] for k in range(n_strategies))
            # Player 2 cannot improve by changing from j
            p2_cannot_improve = all(P2[i, j] >= P2[i, k] for k in range(n_strategies))
            
            if p1_cannot_improve and p2_cannot_improve:
                nash_equilibria.append((i, j, P1[i, j], P2[i, j]))
    
    return nash_equilibria

nash_bos = find_nash_pure(P1_bos, P2_bos)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Payoff visualization
x = np.arange(len(strategies_bos))
width = 0.35

axes[0].bar(x - width/2, P1_bos[:, 0], width, label='P2 plays Opera', alpha=0.8)
axes[0].bar(x + width/2, P1_bos[:, 1], width, label='P2 plays Football', alpha=0.8)
axes[0].set_xlabel('Player 1 Strategy')
axes[0].set_ylabel('Player 1 Payoff')
axes[0].set_title('Player 1 Best Responses')
axes[0].set_xticks(x)
axes[0].set_xticklabels(strategies_bos)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Table with Nash equilibria highlighted
axes[1].axis('off')
table_data_bos = []
for i, s1 in enumerate(strategies_bos):
    row = [s1]
    for j, s2 in enumerate(strategies_bos):
        row.append(f"({P1_bos[i,j]}, {P2_bos[i,j]})")
    table_data_bos.append(row)

table = axes[1].table(cellText=table_data_bos,
                     colLabels=['P1 \\ P2', 'Opera', 'Football'],
                     cellLoc='center',
                     loc='center',
                     bbox=[0.1, 0.4, 0.8, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 3)

# Highlight Nash equilibria
for ne in nash_bos:
    i, j = ne[0], ne[1]
    table[(i+1, j+1)].set_facecolor('#99ff99')

analysis = f"""Battle of the Sexes:

Two Nash Equilibria (in green):
"""
for ne in nash_bos:
    i, j, p1, p2 = ne
    analysis += f"• ({strategies_bos[i]}, {strategies_bos[j]}): ({p1}, {p2})\n"

analysis += f"""
Key Insights:
• Multiple Nash equilibria possible!
• Coordination problem: which equilibrium to play?
• Both prefer being together over being apart
• Asymmetric preferences create conflict
• In practice: communication, commitment, focal points

No dominant strategies here - best response depends
on what the other player does!
"""

axes[1].text(0.5, 0.15, analysis, fontsize=9, family='monospace',
             verticalalignment='top', horizontalalignment='center',
             transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

print(analysis)

## 3. Mixed Strategy Nash Equilibrium

Sometimes there's no pure strategy Nash equilibrium. Players need to **randomize**!

**Mixed Strategy:** A probability distribution over pure strategies.

**Example: Matching Pennies** (zero-sum game)

Two players simultaneously show heads or tails. If they match, P1 wins. If they don't match, P2 wins.

**Finding mixed Nash equilibrium uses calculus!** We maximize expected utility.

In [ ]:
# Matching Pennies
strategies_mp = ['Heads', 'Tails']
P1_mp = np.array([[1, -1], [-1, 1]])   # Zero-sum!
P2_mp = -P1_mp                          # P2's payoffs are negative of P1's

# No pure Nash equilibrium exists (verify)
nash_mp_pure = find_nash_pure(P1_mp, P2_mp)

# Find mixed strategy Nash equilibrium
# Let p = probability P1 plays Heads
# Let q = probability P2 plays Heads

# P1's expected utility when playing Heads: U1(H) = q(1) + (1-q)(-1) = 2q - 1
# P1's expected utility when playing Tails: U1(T) = q(-1) + (1-q)(1) = 1 - 2q
# At equilibrium, P1 must be indifferent: 2q - 1 = 1 - 2q => 4q = 2 => q = 1/2

# Similarly for P2: p = 1/2

p_star = 0.5  # P1 plays Heads with prob 1/2
q_star = 0.5  # P2 plays Heads with prob 1/2

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Payoff table
axes[0, 0].axis('off')
table_data_mp = []
for i, s1 in enumerate(strategies_mp):
    row = [s1]
    for j, s2 in enumerate(strategies_mp):
        row.append(f"({P1_mp[i,j]}, {P2_mp[i,j]})")
    table_data_mp.append(row)

table = axes[0, 0].table(cellText=table_data_mp,
                        colLabels=['P1 \\ P2', 'Heads', 'Tails'],
                        cellLoc='center',
                        loc='center',
                        bbox=[0.1, 0.4, 0.8, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 3)

text1 = f"""Matching Pennies - Zero Sum Game

No pure strategy Nash equilibrium!
(Every cell has an incentive to deviate)

Mixed Strategy Nash Equilibrium:
• P1: Play Heads with p = {p_star}
• P2: Play Heads with q = {q_star}
• Expected payoff for both: 0
"""
axes[0, 0].text(0.5, 0.15, text1, fontsize=10, family='monospace',
               verticalalignment='top', horizontalalignment='center',
               transform=axes[0, 0].transAxes)

# P1's expected utility as function of q (P2's mixing probability)
q_range = np.linspace(0, 1, 100)
U1_heads = 2*q_range - 1      # EU when P1 plays Heads
U1_tails = 1 - 2*q_range      # EU when P1 plays Tails
U1_mixed = np.maximum(U1_heads, U1_tails)  # Best response

axes[0, 1].plot(q_range, U1_heads, 'b-', linewidth=2, label='P1 plays Heads')
axes[0, 1].plot(q_range, U1_tails, 'r-', linewidth=2, label='P1 plays Tails')
axes[0, 1].plot(q_range, U1_mixed, 'g--', linewidth=3, label='P1 Best Response', alpha=0.7)
axes[0, 1].axvline(q_star, color='black', linestyle=':', linewidth=2, label=f'q* = {q_star}')
axes[0, 1].plot(q_star, 0, 'go', markersize=12, label='Equilibrium')
axes[0, 1].set_xlabel('q (Probability P2 plays Heads)')
axes[0, 1].set_ylabel('Expected Utility for P1')
axes[0, 1].set_title('P1 Expected Utility vs P2 Strategy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(0, color='black', linewidth=0.5)

# Best response functions
p_range = np.linspace(0, 1, 100)

# P1's best response to q: BR1(q) = {0 if q < 0.5, any p if q = 0.5, 1 if q > 0.5}
# P2's best response to p: BR2(p) = {0 if p > 0.5, any q if p = 0.5, 1 if p < 0.5}

axes[1, 0].plot([0, 0.5, 0.5], [1, 1, 0], 'b-', linewidth=3, label='P2 Best Response to p')
axes[1, 0].plot([0.5, 0.5, 1], [1, 0, 0], 'b-', linewidth=3)
axes[1, 0].plot([0, 0.5, 0.5], [0, 0, 1], 'r-', linewidth=3, label='P1 Best Response to q')
axes[1, 0].plot([0.5, 0.5, 1], [0, 1, 1], 'r-', linewidth=3)
axes[1, 0].plot(p_star, q_star, 'go', markersize=15, label='Nash Equilibrium', zorder=5)
axes[1, 0].set_xlabel('p (Probability P1 plays Heads)')
axes[1, 0].set_ylabel('q (Probability P2 plays Heads)')
axes[1, 0].set_title('Best Response Functions')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xlim(0, 1)
axes[1, 0].set_ylim(0, 1)

# Calculus derivation
axes[1, 1].axis('off')
derivation = """Finding Mixed Nash Equilibrium (using calculus!):

P1's expected utility:
  U₁(Heads, q) = q·1 + (1-q)·(-1) = 2q - 1
  U₁(Tails, q) = q·(-1) + (1-q)·1 = 1 - 2q

At equilibrium, P1 must be INDIFFERENT:
  U₁(Heads, q*) = U₁(Tails, q*)
  2q* - 1 = 1 - 2q*
  4q* = 2
  q* = 1/2

By symmetry: p* = 1/2

Verification:
  If q = 1/2: U₁(H) = U₁(T) = 0
  → P1 has no incentive to deviate from any p
  If p = 1/2: U₂(H) = U₂(T) = 0
  → P2 has no incentive to deviate from any q

Expected payoff:
  E[U₁] = p·q·1 + p·(1-q)·(-1) + (1-p)·q·(-1) + (1-p)·(1-q)·1
        = 0.5·0.5·1 + 0.5·0.5·(-1) + 0.5·0.5·(-1) + 0.5·0.5·1
        = 0

Key insight: In zero-sum games, mixed strategies
allow both players to guarantee expected value of 0!
"""

axes[1, 1].text(0.05, 0.95, derivation, fontsize=8, family='monospace',
               verticalalignment='top', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()

print(f"Pure strategy Nash equilibria: {nash_mp_pure}")
print(f"Mixed strategy Nash equilibrium: p* = {p_star}, q* = {q_star}")

## 4. Decision Theory Under Uncertainty

**Decision theory** analyzes choices when outcomes are uncertain.

**Expected Utility Theory:**
$$EU(a) = \sum_{s \in S} p(s) \cdot u(a, s)$$

or for continuous outcomes:
$$EU(a) = \int u(a, x) f(x) dx \quad \text{(integration!)}$$

**Risk Attitudes:**
- Risk averse: $u''(x) < 0$ (concave utility)
- Risk neutral: $u''(x) = 0$ (linear utility)
- Risk seeking: $u''(x) > 0$ (convex utility)

In [ ]:
# Expected utility with different risk attitudes

# Gamble: 50% chance of winning $100, 50% chance of winning $0
# vs. Sure thing: $50 for certain

def utility_risk_averse(x):
    """Concave utility: u(x) = sqrt(x)."""
    return np.sqrt(x)

def utility_risk_neutral(x):
    """Linear utility: u(x) = x."""
    return x

def utility_risk_seeking(x):
    """Convex utility: u(x) = x²."""
    return x**2 / 100  # Scaled for visualization

# Outcomes
gamble_outcomes = [0, 100]
gamble_probs = [0.5, 0.5]
sure_thing = 50

# Expected values
EV_gamble = sum(p * x for p, x in zip(gamble_probs, gamble_outcomes))

# Expected utilities
EU_gamble_averse = sum(p * utility_risk_averse(x) for p, x in zip(gamble_probs, gamble_outcomes))
EU_sure_averse = utility_risk_averse(sure_thing)

EU_gamble_neutral = sum(p * utility_risk_neutral(x) for p, x in zip(gamble_probs, gamble_outcomes))
EU_sure_neutral = utility_risk_neutral(sure_thing)

EU_gamble_seeking = sum(p * utility_risk_seeking(x) for p, x in zip(gamble_probs, gamble_outcomes))
EU_sure_seeking = utility_risk_seeking(sure_thing)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x_range = np.linspace(0, 100, 200)

# Risk averse
axes[0].plot(x_range, utility_risk_averse(x_range), 'b-', linewidth=2, label='u(x) = √x')
axes[0].plot([0, 100], [utility_risk_averse(0), utility_risk_averse(100)], 'ro', markersize=10, label='Gamble outcomes')
axes[0].plot([0, 100], [utility_risk_averse(0), utility_risk_averse(100)], 'r--', linewidth=2, alpha=0.5, label='Expected utility line')
axes[0].plot(EV_gamble, EU_gamble_averse, 'gs', markersize=12, label=f'EU(Gamble) = {EU_gamble_averse:.2f}')
axes[0].plot(sure_thing, EU_sure_averse, 'ms', markersize=12, label=f'U(Sure) = {EU_sure_averse:.2f}')
axes[0].axvline(sure_thing, color='gray', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Wealth ($)')
axes[0].set_ylabel('Utility')
axes[0].set_title(f'Risk Averse: Prefers Sure Thing\n(Concave, u\'\'(x) < 0)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Risk neutral
axes[1].plot(x_range, utility_risk_neutral(x_range), 'b-', linewidth=2, label='u(x) = x')
axes[1].plot([0, 100], [utility_risk_neutral(0), utility_risk_neutral(100)], 'ro', markersize=10, label='Gamble outcomes')
axes[1].plot(EV_gamble, EU_gamble_neutral, 'gs', markersize=12, label=f'EU(Gamble) = {EU_gamble_neutral:.2f}')
axes[1].plot(sure_thing, EU_sure_neutral, 'ms', markersize=12, label=f'U(Sure) = {EU_sure_neutral:.2f}')
axes[1].axvline(sure_thing, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Wealth ($)')
axes[1].set_ylabel('Utility')
axes[1].set_title(f'Risk Neutral: Indifferent\n(Linear, u\'\'(x) = 0)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Risk seeking
axes[2].plot(x_range, utility_risk_seeking(x_range), 'b-', linewidth=2, label='u(x) = x²/100')
axes[2].plot([0, 100], [utility_risk_seeking(0), utility_risk_seeking(100)], 'ro', markersize=10, label='Gamble outcomes')
axes[2].plot([0, 100], [utility_risk_seeking(0), utility_risk_seeking(100)], 'r--', linewidth=2, alpha=0.5)
axes[2].plot(EV_gamble, EU_gamble_seeking, 'gs', markersize=12, label=f'EU(Gamble) = {EU_gamble_seeking:.2f}')
axes[2].plot(sure_thing, EU_sure_seeking, 'ms', markersize=12, label=f'U(Sure) = {EU_sure_seeking:.2f}')
axes[2].axvline(sure_thing, color='gray', linestyle=':', alpha=0.5)
axes[2].set_xlabel('Wealth ($)')
axes[2].set_ylabel('Utility')
axes[2].set_title(f'Risk Seeking: Prefers Gamble\n(Convex, u\'\'(x) > 0)')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Decision Under Uncertainty:")
print(f"\nGamble: 50% × $0 + 50% × $100")
print(f"Expected value: ${EV_gamble}")
print(f"Sure thing: ${sure_thing}")
print(f"\nRisk Averse (u = √x):")
print(f"  EU(Gamble) = {EU_gamble_averse:.4f} < U(Sure) = {EU_sure_averse:.4f} → Prefers sure thing")
print(f"\nRisk Neutral (u = x):")
print(f"  EU(Gamble) = {EU_gamble_neutral:.4f} = U(Sure) = {EU_sure_neutral:.4f} → Indifferent")
print(f"\nRisk Seeking (u = x²):")
print(f"  EU(Gamble) = {EU_gamble_seeking:.4f} > U(Sure) = {EU_sure_seeking:.4f} → Prefers gamble")
print(f"\nKey: Second derivative u''(x) determines risk attitude!")

## 5. Continuous Strategy Games - Using Calculus!

When strategies are continuous (e.g., prices, quantities), we use **calculus** to find Nash equilibrium.

**Cournot Competition:** Two firms choose quantities simultaneously.

**Finding equilibrium:**
1. Write each firm's profit function: $\pi_i(q_i, q_j)$
2. Find best response by taking derivative: $\frac{\partial \pi_i}{\partial q_i} = 0$
3. Solve system of best response functions for Nash equilibrium

In [ ]:
# Cournot Duopoly
# Market demand: P(Q) = a - b*Q where Q = q1 + q2
# Firm costs: C(q) = c*q (constant marginal cost)

a = 100  # Demand intercept
b = 1    # Demand slope
c = 10   # Marginal cost

def price(q1, q2):
    """Market price given quantities."""
    Q = q1 + q2
    return max(0, a - b*Q)

def profit_firm1(q1, q2):
    """Firm 1 profit."""
    return q1 * (price(q1, q2) - c)

def profit_firm2(q1, q2):
    """Firm 2 profit (by symmetry, same as firm 1)."""
    return q2 * (price(q1, q2) - c)

# Best response functions (derived using calculus)
# π₁ = q₁(a - b(q₁ + q₂) - c) = q₁(a - c - bq₁ - bq₂)
# ∂π₁/∂q₁ = a - c - 2bq₁ - bq₂ = 0
# => q₁ = (a - c - bq₂)/(2b)

def best_response_1(q2):
    """Firm 1 best response to firm 2's quantity."""
    return max(0, (a - c - b*q2) / (2*b))

def best_response_2(q1):
    """Firm 2 best response to firm 1's quantity."""
    return max(0, (a - c - b*q1) / (2*b))

# Nash equilibrium: solve q₁ = BR₁(q₂) and q₂ = BR₂(q₁)
# By symmetry: q₁* = q₂* = q*
# q* = (a - c - bq*)/(2b)
# 2bq* = a - c - bq*
# 3bq* = a - c
# q* = (a - c)/(3b)

q_nash = (a - c) / (3*b)
Q_nash = 2 * q_nash
P_nash = price(q_nash, q_nash)
profit_nash = profit_firm1(q_nash, q_nash)

# For comparison: competitive equilibrium (P = MC)
Q_competitive = (a - c) / b
P_competitive = c

# Monopoly (maximize joint profit)
Q_monopoly = (a - c) / (2*b)
P_monopoly = (a + c) / 2

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Best response functions
q_range = np.linspace(0, 50, 200)
br1 = np.array([best_response_1(q2) for q2 in q_range])
br2 = np.array([best_response_2(q1) for q1 in q_range])

axes[0, 0].plot(q_range, br1, 'b-', linewidth=2, label='BR₁(q₂)')
axes[0, 0].plot(br2, q_range, 'r-', linewidth=2, label='BR₂(q₁)')
axes[0, 0].plot(q_nash, q_nash, 'go', markersize=15, label=f'Nash Eq: ({q_nash:.1f}, {q_nash:.1f})', zorder=5)
axes[0, 0].set_xlabel('q₁ (Firm 1 quantity)')
axes[0, 0].set_ylabel('q₂ (Firm 2 quantity)')
axes[0, 0].set_title('Best Response Functions')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xlim(0, 50)
axes[0, 0].set_ylim(0, 50)

# Profit surface for Firm 1
q1_grid = np.linspace(0, 50, 50)
q2_grid = np.linspace(0, 50, 50)
Q1, Q2 = np.meshgrid(q1_grid, q2_grid)
Profit1 = np.vectorize(profit_firm1)(Q1, Q2)

ax3d = fig.add_subplot(2, 2, 2, projection='3d')
surf = ax3d.plot_surface(Q1, Q2, Profit1, cmap='viridis', alpha=0.7)
ax3d.scatter([q_nash], [q_nash], [profit_nash], color='red', s=100, label='Nash Eq')
ax3d.set_xlabel('q₁')
ax3d.set_ylabel('q₂')
ax3d.set_zlabel('π₁')
ax3d.set_title('Firm 1 Profit Surface')
fig.colorbar(surf, ax=ax3d, shrink=0.5)

# Market outcomes comparison
outcomes = ['Monopoly', 'Cournot (Nash)', 'Competitive']
quantities = [Q_monopoly, Q_nash, Q_competitive]
prices = [P_monopoly, P_nash, P_competitive]

x = np.arange(len(outcomes))
width = 0.35

axes[1, 0].bar(x - width/2, quantities, width, label='Quantity', alpha=0.8)
axes[1, 0].bar(x + width/2, prices, width, label='Price', alpha=0.8)
axes[1, 0].set_ylabel('Value')
axes[1, 0].set_title('Market Outcomes Comparison')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(outcomes)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Calculus derivation
axes[1, 1].axis('off')
derivation = f"""Cournot Competition (using calculus!):

Setup:
  Demand: P = {a} - {b}Q, where Q = q₁ + q₂
  Cost: C(qᵢ) = {c}qᵢ

Firm 1's profit:
  π₁(q₁, q₂) = q₁·P(Q) - C(q₁)
             = q₁·({a} - {b}(q₁ + q₂)) - {c}q₁
             = q₁({a} - {c} - {b}q₁ - {b}q₂)

First-order condition (FOC):
  ∂π₁/∂q₁ = {a} - {c} - {2*b}q₁ - {b}q₂ = 0

Best response function:
  q₁ = ({a} - {c} - {b}q₂) / {2*b}

By symmetry: q₂ = ({a} - {c} - {b}q₁) / {2*b}

Nash equilibrium (solve simultaneously):
  q₁* = q₂* = ({a} - {c}) / {3*b} = {q_nash:.2f}
  Q* = {Q_nash:.2f}
  P* = {P_nash:.2f}
  π₁* = π₂* = {profit_nash:.2f}

Second-order condition:
  ∂²π₁/∂q₁² = -{2*b} < 0 ✓ (maximum confirmed!)

Comparisons:
  Monopoly: Q = {Q_monopoly:.2f}, P = {P_monopoly:.2f}
  Cournot:  Q = {Q_nash:.2f}, P = {P_nash:.2f}
  Competitive: Q = {Q_competitive:.2f}, P = {P_competitive:.2f}

More competition → Higher Q, Lower P!
"""

axes[1, 1].text(0.05, 0.95, derivation, fontsize=8, family='monospace',
               verticalalignment='top', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()

print(derivation)

## 6. Evolutionary Game Theory

**Evolutionary Game Theory** studies strategy evolution in populations.

**Replicator Dynamics** (uses differential equations!):
$$\frac{dx_i}{dt} = x_i \left[ u_i(x) - \bar{u}(x) \right]$$

where $x_i$ is the fraction playing strategy $i$, $u_i$ is the payoff to strategy $i$, and $\bar{u}$ is the average payoff.

**Evolutionarily Stable Strategy (ESS):** A strategy that, if adopted by most of the population, cannot be invaded by any alternative strategy.

In [ ]:
# Hawk-Dove game (evolutionary)
# Hawk: aggressive, fights for resource
# Dove: passive, shares or retreats

# Payoffs
V = 4  # Value of resource
C = 6  # Cost of fighting

# Payoff matrix
# (Hawk, Hawk): (V-C)/2 each (fight, split resource minus cost)
# (Hawk, Dove): V to Hawk, 0 to Dove
# (Dove, Hawk): 0 to Dove, V to Hawk
# (Dove, Dove): V/2 each (share peacefully)

payoff_HH = (V - C) / 2
payoff_HD = V
payoff_DH = 0
payoff_DD = V / 2

# Expected payoff to Hawk when fraction p plays Hawk:
# u_H(p) = p·(V-C)/2 + (1-p)·V
def payoff_hawk(p):
    return p * payoff_HH + (1 - p) * payoff_HD

# Expected payoff to Dove:
# u_D(p) = p·0 + (1-p)·V/2
def payoff_dove(p):
    return p * payoff_DH + (1 - p) * payoff_DD

# Average payoff in population:
def avg_payoff(p):
    return p * payoff_hawk(p) + (1 - p) * payoff_dove(p)

# Replicator dynamics: dp/dt = p(u_H - u_avg)
def replicator_dynamics(p):
    return p * (payoff_hawk(p) - avg_payoff(p))

# Find ESS: set u_H(p*) = u_D(p*)
# p*(V-C)/2 + (1-p*)V = (1-p*)V/2
# p*(V-C)/2 + V - p*V = V/2 - p*V/2
# p*(V-C)/2 + V - p*V = V/2 - p*V/2
# p*(V-C)/2 - p*V + p*V/2 = V/2 - V
# p*[(V-C)/2 - V + V/2] = -V/2
# p*[-C/2] = -V/2
# p* = V/C

p_ess = V / C

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Payoff functions
p_range = np.linspace(0, 1, 100)
u_H = np.array([payoff_hawk(p) for p in p_range])
u_D = np.array([payoff_dove(p) for p in p_range])
u_avg = np.array([avg_payoff(p) for p in p_range])

axes[0, 0].plot(p_range, u_H, 'r-', linewidth=2, label='Hawk payoff')
axes[0, 0].plot(p_range, u_D, 'b-', linewidth=2, label='Dove payoff')
axes[0, 0].plot(p_range, u_avg, 'g--', linewidth=2, label='Average payoff')
axes[0, 0].axvline(p_ess, color='black', linestyle=':', linewidth=2, label=f'ESS: p* = {p_ess:.3f}')
axes[0, 0].plot(p_ess, payoff_hawk(p_ess), 'ko', markersize=12)
axes[0, 0].set_xlabel('p (Fraction playing Hawk)')
axes[0, 0].set_ylabel('Expected Payoff')
axes[0, 0].set_title(f'Hawk-Dove Game (V={V}, C={C})')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Replicator dynamics
dp_dt = np.array([replicator_dynamics(p) for p in p_range])

axes[0, 1].plot(p_range, dp_dt, 'purple', linewidth=2)
axes[0, 1].axhline(0, color='black', linewidth=1)
axes[0, 1].axvline(p_ess, color='black', linestyle=':', linewidth=2, label=f'Stable eq: p* = {p_ess:.3f}')
axes[0, 1].fill_between(p_range, 0, dp_dt, where=(dp_dt > 0), alpha=0.3, color='green', label='p increasing')
axes[0, 1].fill_between(p_range, 0, dp_dt, where=(dp_dt < 0), alpha=0.3, color='red', label='p decreasing')
axes[0, 1].set_xlabel('p (Fraction playing Hawk)')
axes[0, 1].set_ylabel('dp/dt (Rate of change)')
axes[0, 1].set_title('Replicator Dynamics')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Simulation of population evolution
def simulate_replicator(p0, dt=0.01, steps=500):
    """Simulate replicator dynamics."""
    p = p0
    history = [p]
    
    for _ in range(steps):
        dp = replicator_dynamics(p) * dt
        p = np.clip(p + dp, 0, 1)  # Keep in [0, 1]
        history.append(p)
    
    return np.array(history)

# Multiple trajectories
initial_conditions = [0.1, 0.3, 0.5, 0.7, 0.9]
colors = plt.cm.viridis(np.linspace(0, 1, len(initial_conditions)))

for p0, color in zip(initial_conditions, colors):
    trajectory = simulate_replicator(p0)
    axes[1, 0].plot(trajectory, linewidth=2, label=f'p(0) = {p0}', color=color)

axes[1, 0].axhline(p_ess, color='red', linestyle='--', linewidth=2, label=f'ESS: p* = {p_ess:.3f}')
axes[1, 0].set_xlabel('Time Step')
axes[1, 0].set_ylabel('p (Fraction playing Hawk)')
axes[1, 0].set_title('Population Evolution Over Time')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, alpha=0.3)

# Analysis
axes[1, 1].axis('off')
analysis = f"""Hawk-Dove Evolutionary Game:

Parameters:
  V = {V} (value of resource)
  C = {C} (cost of fighting)

Payoff Matrix:
              Hawk         Dove
  Hawk    {payoff_HH:6.1f}    {payoff_HD:6.1f}
  Dove    {payoff_DH:6.1f}    {payoff_DD:6.1f}

Expected payoffs:
  u_H(p) = p·{payoff_HH:.1f} + (1-p)·{payoff_HD:.1f}
  u_D(p) = p·{payoff_DH:.1f} + (1-p)·{payoff_DD:.1f}

ESS (Evolutionarily Stable Strategy):
  Set u_H(p*) = u_D(p*):
  p*·{payoff_HH:.1f} + (1-p*)·{payoff_HD:.1f} = (1-p*)·{payoff_DD:.1f}
  
  Solving: p* = V/C = {V}/{C} = {p_ess:.4f}

Interpretation:
  • Mixed population at equilibrium!
  • {p_ess*100:.1f}% Hawks, {(1-p_ess)*100:.1f}% Doves
  • If p < p*: Hawks do better → p increases
  • If p > p*: Doves do better → p decreases
  • Stable polymorphic equilibrium

Replicator Dynamics (differential equation!):
  dp/dt = p(u_H - ū)
  Stable fixed point at p* = {p_ess:.4f}

Note: When C > V (fighting too costly),
pure Dove is ESS. When C < V, analysis
differs.
"""

axes[1, 1].text(0.05, 0.95, analysis, fontsize=8, family='monospace',
               verticalalignment='top', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()

print(analysis)

## 7. Application: First-Price Sealed-Bid Auction

**Auction Theory** is a major application of game theory.

**Setup:**
- $n$ bidders with private valuations $v_i \sim U[0, 1]$
- Each submits sealed bid $b_i$
- Highest bidder wins, pays their bid

**Finding equilibrium bidding strategy uses calculus!**

In symmetric equilibrium with 2 bidders, optimal bid: $b(v) = \frac{v}{2}$ (bid half your value)

In [ ]:
# First-price sealed-bid auction
# Derivation of optimal bidding strategy

# With 2 bidders, valuations uniform on [0, 1]
# Bidder with value v wins if b(v) > b(v_other)
# Assuming linear strategy b(v) = αv:
# Prob(win) = Prob(αv > αv_other) = Prob(v > v_other) = v

# Expected utility:
# EU(b|v) = Prob(win) × (v - b) = v × (v - b)

# But in equilibrium, both use same strategy b(v) = αv
# So if I bid b while others use b(v) = αv:
# Prob(win) = Prob(b > αv_other) = b/α

# EU(b|v) = (b/α)(v - b)
# FOC: dEU/db = (v - b)/α - b/α = (v - 2b)/α = 0
# => b = v/2
# So α = 1/2

def optimal_bid_2_bidders(v):
    """Optimal bid with 2 bidders."""
    return v / 2

def optimal_bid_n_bidders(v, n):
    """Optimal bid with n bidders."""
    return ((n - 1) / n) * v

# Simulation
n_auctions = 10000
n_bidders = 2

revenues = []
winner_values = []
winner_payoffs = []

for _ in range(n_auctions):
    # Draw valuations
    valuations = np.random.uniform(0, 1, n_bidders)
    # Optimal bids
    bids = optimal_bid_n_bidders(valuations, n_bidders)
    # Winner
    winner_idx = np.argmax(bids)
    winning_bid = bids[winner_idx]
    winner_value = valuations[winner_idx]
    
    revenues.append(winning_bid)
    winner_values.append(winner_value)
    winner_payoffs.append(winner_value - winning_bid)

revenues = np.array(revenues)
winner_values = np.array(winner_values)
winner_payoffs = np.array(winner_payoffs)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Bidding strategies
v_range = np.linspace(0, 1, 100)
for n in [2, 3, 4, 5]:
    bids = optimal_bid_n_bidders(v_range, n)
    axes[0, 0].plot(v_range, bids, linewidth=2, label=f'n = {n}')

axes[0, 0].plot(v_range, v_range, 'k--', linewidth=1, alpha=0.5, label='Truthful bidding')
axes[0, 0].set_xlabel('Valuation (v)')
axes[0, 0].set_ylabel('Optimal Bid b(v)')
axes[0, 0].set_title('Optimal Bidding Strategies')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Revenue distribution
axes[0, 1].hist(revenues, bins=50, density=True, alpha=0.7, edgecolor='black')
axes[0, 1].axvline(revenues.mean(), color='red', linestyle='--', linewidth=2, 
                   label=f'Mean = {revenues.mean():.3f}')
# Theoretical: with 2 bidders, revenue = max(v1, v2)/2
# E[max(v1, v2)] = 2/3, so E[revenue] = 1/3
axes[0, 1].axvline(1/3, color='green', linestyle=':', linewidth=2, label='Theoretical = 1/3')
axes[0, 1].set_xlabel('Revenue')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title(f'Auction Revenue Distribution (n={n_bidders})')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Winner's value vs bid
axes[1, 0].scatter(winner_values, revenues, alpha=0.1, s=5)
axes[1, 0].plot(v_range, optimal_bid_n_bidders(v_range, n_bidders), 'r-', 
                linewidth=2, label=f'b(v) = v/{n_bidders}')
axes[1, 0].set_xlabel('Winner Valuation')
axes[1, 0].set_ylabel('Winning Bid (Revenue)')
axes[1, 0].set_title('Winner Value vs Bid')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Derivation
axes[1, 1].axis('off')
derivation = f"""First-Price Auction Equilibrium (using calculus!):

Setup:
  • n = {n_bidders} bidders
  • Private valuations: vᵢ ~ Uniform[0, 1]
  • Sealed bids, highest wins, pays their bid

Symmetric equilibrium: b(v) = αv

For bidder with value v bidding b:
  Prob(win) = Prob(b > b(vⱼ) ∀j≠i)
            = Prob(b > αvⱼ ∀j≠i)
            = (b/α)^(n-1)  [for uniform valuations]

Expected utility:
  EU(b|v) = Prob(win) × (v - b)
          = (b/α)^(n-1) × (v - b)

First-order condition:
  dEU/db = (n-1)(b/α)^(n-2) · (1/α) · (v-b) - (b/α)^(n-1) = 0
  (n-1)(v - b) - b = 0
  (n-1)v = nb
  b = [(n-1)/n]v

For n = {n_bidders}: b(v) = {(n_bidders-1)/n_bidders}v = v/{n_bidders/(n_bidders-1):.0f}

Simulation results ({n_auctions:,} auctions):
  Mean revenue: {revenues.mean():.4f}
  Theoretical:  {1/3:.4f}
  Mean winner payoff: {winner_payoffs.mean():.4f}

Key insights:
  • Bid shading: b(v) < v (never bid true value!)
  • More bidders → less shading (more competition)
  • As n → ∞: b(v) → v (competitive limit)
  • Revenue equivalence theorem: expected revenue
    same as second-price auction!
"""

axes[1, 1].text(0.05, 0.95, derivation, fontsize=8, family='monospace',
               verticalalignment='top', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()

print(derivation)

## 8. Practice Problems

### Problem 1: Finding Nash Equilibrium

Consider the following game:

```
        Player 2
        L    R
P1  T  (3,2) (1,3)
    B  (2,1) (4,4)
```

Find all pure strategy Nash equilibria.

In [ ]:
# Solution
P1_payoffs_prob1 = np.array([[3, 1], [2, 4]])
P2_payoffs_prob1 = np.array([[2, 3], [1, 4]])

nash_prob1 = find_nash_pure(P1_payoffs_prob1, P2_payoffs_prob1)

print("Problem 1: Nash Equilibria")
strategies_prob1 = [['T', 'L'], ['T', 'R'], ['B', 'L'], ['B', 'R']]
for i, j, p1, p2 in nash_prob1:
    print(f"  ({['T', 'B'][i]}, {['L', 'R'][j]}): Payoffs = ({p1}, {p2})")

print("\nVerification:")
print("At (B, R): Both players get 4")
print("  P1 cannot improve: 4 > 1 (switching to T)")
print("  P2 cannot improve: 4 > 3 (switching to L)")
print("  → (B, R) is a Nash equilibrium ✓")

### Problem 2: Continuous Strategy with Calculus

Two firms compete by choosing prices $p_1$ and $p_2$.

Demand functions:
- $q_1(p_1, p_2) = 100 - 2p_1 + p_2$
- $q_2(p_1, p_2) = 100 - 2p_2 + p_1$

Cost: $C_i(q_i) = 10q_i$

Find the Nash equilibrium prices using calculus.

In [ ]:
# Solution using calculus
# Profit for firm 1: π₁ = p₁·q₁ - C₁
#                      = p₁(100 - 2p₁ + p₂) - 10(100 - 2p₁ + p₂)
#                      = p₁(100 - 2p₁ + p₂) - 1000 + 20p₁ - 10p₂
#                      = 100p₁ - 2p₁² + p₁p₂ + 20p₁ - 10p₂ - 1000

# FOC: ∂π₁/∂p₁ = 100 - 4p₁ + p₂ + 20 = 0
#      120 - 4p₁ + p₂ = 0
#      p₁ = (120 + p₂)/4

# By symmetry: p₂ = (120 + p₁)/4

# Solving: p₁ = (120 + p₂)/4 = (120 + (120 + p₁)/4)/4
#          4p₁ = 120 + (120 + p₁)/4
#          16p₁ = 480 + 120 + p₁
#          15p₁ = 600
#          p₁ = 40

p1_nash = 40
p2_nash = 40  # By symmetry

q1_nash = 100 - 2*p1_nash + p2_nash
q2_nash = 100 - 2*p2_nash + p1_nash

profit1 = p1_nash * q1_nash - 10 * q1_nash
profit2 = p2_nash * q2_nash - 10 * q2_nash

print("Problem 2: Bertrand Competition with Differentiated Products")
print(f"\nNash equilibrium:")
print(f"  p₁* = ${p1_nash}")
print(f"  p₂* = ${p2_nash}")
print(f"  q₁* = {q1_nash} units")
print(f"  q₂* = {q2_nash} units")
print(f"  π₁* = ${profit1}")
print(f"  π₂* = ${profit2}")

print(f"\nDerivation:")
print(f"  π₁ = p₁(100 - 2p₁ + p₂) - 10(100 - 2p₁ + p₂)")
print(f"  ∂π₁/∂p₁ = 120 - 4p₁ + p₂ = 0")
print(f"  Best response: p₁ = (120 + p₂)/4")
print(f"  By symmetry and solving: p₁* = p₂* = 40")

## Summary

**Key Takeaways:**

1. **Normal Form Games** represent strategic interactions
   - Players, strategies, payoffs
   - Dominant strategies simplify analysis

2. **Nash Equilibrium** is the central solution concept
   - No player can improve by unilaterally deviating
   - Can be pure or mixed strategies
   - Multiple equilibria possible

3. **Mixed Strategies** require indifference conditions
   - Set expected utilities equal
   - Solve for equilibrium probabilities
   - Important in zero-sum games

4. **Decision Theory** uses expected utility
   - $EU = \int u(x) f(x) dx$ (integration!)
   - Risk attitude determined by $u''(x)$
   - Concave = risk averse, Linear = risk neutral, Convex = risk seeking

5. **Continuous Strategies** require calculus
   - Find best responses: $\frac{\partial \pi_i}{\partial x_i} = 0$
   - Solve system of FOCs for Nash equilibrium
   - Examples: Cournot, Bertrand, auctions

6. **Evolutionary Game Theory** uses differential equations
   - Replicator dynamics: $\frac{dx}{dt} = x(u - \bar{u})$
   - ESS = stable fixed point
   - Applications in biology, social dynamics

7. **Auction Theory** applies optimization
   - Optimal bidding: maximize expected utility
   - First-order conditions determine bid functions
   - Revenue equivalence theorem

**Connection to Calculus:**
Game theory extensively uses:
- **Derivatives** to find best responses and Nash equilibria
- **Integration** for expected utilities and probabilities
- **Differential equations** in evolutionary dynamics
- **Optimization** throughout decision-making and strategy selection

Game theory is applied mathematics using calculus to understand strategic behavior!